# Placar e análise das falhas

As respostas e tempos foram produzidos na máquina da autora. Este notebook executa a análise dos arquivos congelados e a leitura da revisão assistida por IA, sujeita à conferência da autora. Validade de JSON, correção do conteúdo e aceitação pelo validador são métricas distintas.


In [1]:
from pathlib import Path
import json
import sys

RAIZ = Path.cwd()
if not (RAIZ / "base" / "conhecimento.json").exists():
    RAIZ = RAIZ.parent
assert (RAIZ / "base" / "conhecimento.json").exists(), "Abra a partir da raiz do projeto ou da pasta notebooks."
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

def ler_json(caminho):
    return json.loads((RAIZ / caminho).read_text(encoding="utf-8-sig"))


In [2]:
avaliacao = ler_json("avaliacao/avaliacao_congelada_20261114_160302_507373.json")
revisao = ler_json("avaliacao/revisao_assistida_experimento_03.json")
for chave, item in revisao["resumo"].items():
    print("\n", chave)
    print(json.dumps(item, ensure_ascii=False, indent=2))
print("\nRecall@1:", avaliacao["resumo_automatico"]["recall_1"])
print("Recall@3:", avaliacao["resumo_automatico"]["recall_3"])



 a_sem_contexto
{
  "total": 12,
  "json_validos": 12,
  "validacoes_formais_aprovadas": 0,
  "conteudo_correto_completo": 5,
  "aproveitaveis_criterio_estrito": 0,
  "conteudo_sem_suporte_na_resposta": 4,
  "fonte_ou_citacao_invalida": 7,
  "tempo_medio_geracao_segundos": 8.696682991666648
}

 b_rag_simples
{
  "total": 12,
  "json_validos": 12,
  "validacoes_formais_aprovadas": 3,
  "conteudo_correto_completo": 9,
  "aproveitaveis_criterio_estrito": 3,
  "conteudo_sem_suporte_na_resposta": 1,
  "fonte_ou_citacao_invalida": 2,
  "tempo_medio_geracao_segundos": 9.45063491666671
}

 c_lora_rag
{
  "total": 12,
  "json_validos": 12,
  "validacoes_formais_aprovadas": 11,
  "conteudo_correto_completo": 10,
  "aproveitaveis_criterio_estrito": 9,
  "conteudo_sem_suporte_na_resposta": 1,
  "fonte_ou_citacao_invalida": 1,
  "tempo_medio_geracao_segundos": 9.707015425000009
}

Recall@1: 0.8333333333333334
Recall@3: 0.8333333333333334


In [3]:
chaves = ["a_sem_contexto", "b_rag_simples", "c_lora_rag"]
for chave in chaves:
    registros = [item for item in revisao["resultados"] if item["configuracao"] == chave]
    calculado = sum(item["conteudo_correto_completo"] and item["validacao_formal_aprovada"] for item in registros)
    assert calculado == revisao["resumo"][chave]["aproveitaveis_criterio_estrito"]
    casos_ausentes = {caso["id"] for caso in avaliacao["resultados"] if not caso["fontes"]}
    corretas = sum(item["conteudo_correto_completo"] for item in registros if item["id"] in casos_ausentes)
    print(chave, "| critério estrito", calculado, "/ 12 | ausência reconhecida no texto", corretas, "/ 6")


a_sem_contexto | critério estrito 0 / 12 | ausência reconhecida no texto 5 / 6
b_rag_simples | critério estrito 3 / 12 | ausência reconhecida no texto 6 / 6
c_lora_rag | critério estrito 9 / 12 | ausência reconhecida no texto 6 / 6


## Todas as saídas do modelo

A célula exibe as 36 respostas originais, o erro formal e a observação da revisão. Os casos são mantidos mesmo quando reduzem a pontuação. Nenhuma saída foi substituída pelo gabarito.


In [4]:
indice = {(item["id"], item["configuracao"]): item for item in revisao["resultados"]}
for caso in avaliacao["resultados"]:
    print("\n", caso["id"], caso["pergunta"])
    print("Gabarito:", caso["gabarito"])
    for chave in chaves:
        gerado = json.loads(caso[chave]["saida_original"])
        print(chave, "|", gerado["status"], "|", gerado["resposta"])
        print("Erro formal:", caso[chave]["erro_validacao"])
        print("Revisão:", indice[(caso["id"], chave)]["observacao"])



 j001 Um familiar desapareceu há poucas horas. Devo aguardar até amanhã para registrar o boletim?
Gabarito: Não é necessário esperar 24 horas para comunicar o desaparecimento e registrar o boletim de ocorrência.
a_sem_contexto | respondido | Resposta fundamentada nos documentos.
Erro formal: O modelo copiou o texto de exemplo do formato.
Revisão: Texto de exemplo e identificador de fonte inexistente.
b_rag_simples | respondido | Resposta fundamentada nos documentos.
Erro formal: O modelo copiou o texto de exemplo do formato.
Revisão: Citação real, mas a resposta copia o texto de exemplo.
c_lora_rag | respondido | Não é necessário esperar 24 horas para comunicar o desaparecimento e registrar o boletim de ocorrência.
Erro formal: None
Revisão: Resposta e citação preservam a dispensa de espera.

 j002 Já fiz a comunicação pelo SINAL DESAPARECIDOS. Isso elimina a necessidade de registrar ocorrência?
Gabarito: A comunicação pelo SINAL DESAPARECIDOS não dispensa o registro de ocorrência na 

## Conclusão delimitada

O critério estrito obteve 0, 3 e 9 saídas aproveitáveis em 12. O conteúdo sem suporte na resposta ocorreu em 4, 1 e 1 casos, respectivamente. O LoRA reduziu essa ocorrência em relação ao modelo sem contexto neste conjunto, mas não em relação ao RAG simples. As categorias podem se sobrepor e foram definidas após inspeção, sem pré-registro ou avaliador humano independente.

O baseline sem contexto mantém a instrução de usar documentos mesmo sem recebê-los, o que limita a comparação com uso geral do modelo. O limiar falha em uma pergunta respondível. A API se abstém antes de gerar em busca vazia, diferente do avaliador. A seleção da versão 03 e a integração local não demonstram segurança geral, conformidade integral à LGPD ou desempenho em desaparecimentos reais.
